# Baseline Inference: kg-axel (Single-Shot Generation)

**Purpose**: Run baseline kg-axel system (CoT + only_paths) untuk establish baseline metrics

**Output**: `output/baseline_kg_axel.csv`

## Baseline Configuration:
- System: kg-axel
- Prompt: Chain-of-Thought (CoT)
- Schema: only_paths
- Generation: Single-shot (no refinement)

## Output Columns:
| Column | Description |
|--------|-------------|
| question_id | Question identifier |
| question | Natural language question |
| ground_truth | Ground truth Cypher query |
| complexity | Easy/Medium/Hard |
| reasoning_level | Fakta Eksplisit/Implisit |
| sublevel | Nodes/One-hop/Multi-hop |
| generated_query | Baseline generated query |
| execution_result | Query execution outcome |
| execution_success | Boolean: query executed successfully |
| is_empty_result | Boolean: query returned empty results |
| pass_at_1 | Boolean: output matches GT |
| total_tokens | Tokens used |
| input_tokens | Input tokens |
| output_tokens | Output tokens |
| elapsed_time | Processing time (seconds) |

## 1. Setup

In [ ]:
import sys
import os
import time
import csv
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print(f"Notebook started: {datetime.now()}")
print(f"Working directory: {Path.cwd()}")

## 2. Import Modules

In [ ]:
# Baseline system (kg-axel)
from baseline.kg_axel_generator import KGAxelGenerator

# Database executor
from system.graph_executor import GraphExecutor

# Utilities
from utils.schema_loader import load_schema
from utils.prompt_loader import load_prompt_template

print("Modules imported successfully")

## 3. Configuration

In [ ]:
# Experiment configuration
CONFIG = {
    "name": "baseline_kg_axel",
    "model": os.getenv("DEFAULT_MODEL", "qwen/qwen-2.5-coder-32b-instruct"),
    "prompt_type": "cot",  # Chain-of-Thought
    "schema_type": "only_paths",  # Best config from kg-axel
    "temperature": float(os.getenv("TEMPERATURE", 0.0)),
    "max_tokens": int(os.getenv("MAX_TOKENS", 512)),
    "rate_limit_delay": float(os.getenv("RATE_LIMIT_DELAY", 2.0)),
    "batch_size": int(os.getenv("BATCH_SIZE", 10)),
    "batch_pause": float(os.getenv("BATCH_PAUSE", 15.0)),
}

# Paths
OUTPUT_DIR = Path.cwd().parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILENAME = f"{CONFIG['name']}.csv"
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILENAME

print("Baseline Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nOutput: {OUTPUT_PATH}")

## 4. Load Data

In [ ]:
# Load ground truth questions
gt_file = Path.cwd().parent / "data" / "ground_truth" / "ground_truth_52.csv"

questions = []
with open(gt_file, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        questions.append({
            "id": i + 1,
            "question": row["Pertanyaan"],
            "ground_truth": row["Cypher Query"],
            "complexity": row["Tingkat Kompleksitas"],
            "reasoning_level": row["Tingkat Penalaran"],
            "sublevel": row["Sublevel"]
        })

print(f"Loaded {len(questions)} questions")

# Show distribution
from collections import Counter
print(f"\nBy Complexity: {dict(Counter(q['complexity'] for q in questions))}")
print(f"By Reasoning: {dict(Counter(q['reasoning_level'] for q in questions))}")
print(f"By Sublevel: {dict(Counter(q['sublevel'] for q in questions))}")

## 5. Load Schema

In [ ]:
# Load schema
schema = load_schema(CONFIG["schema_type"])

print(f"Schema loaded: {CONFIG['schema_type']} ({len(schema)} chars)")
print(f"\nSchema preview:\n{schema[:300]}...")

## 6. Initialize Baseline System

In [ ]:
# Initialize kg-axel generator (CoT + only_paths)
baseline = KGAxelGenerator(
    model=CONFIG["model"],
    temperature=CONFIG["temperature"],
    max_tokens=CONFIG["max_tokens"],
    prompt_type=CONFIG["prompt_type"]
)

# Initialize graph executor (optional - for execution validation)
try:
    executor = GraphExecutor()
    print("Graph executor initialized successfully")
except Exception as e:
    print(f"Warning: Could not initialize graph executor - {e}")
    print("Will skip execution validation")
    executor = None

print(f"\nBaseline system initialized")
print(f"  Model: {CONFIG['model']}")
print(f"  Prompt: {CONFIG['prompt_type']}")
print(f"  Schema: {CONFIG['schema_type']}")

## 7. Run Baseline Inference

In [ ]:
from IPython.display import clear_output

# Storage for results
results = []

def update_display(current, total, q_id, success, tokens):
    """Update progress display."""
    clear_output(wait=True)
    pct = current / total * 100
    
    # Calculate running statistics
    pass_at_1_count = sum(1 for r in results if r.get("pass_at_1"))
    exec_success_count = sum(1 for r in results if r.get("execution_success"))
    total_tokens_so_far = sum(r.get("total_tokens", 0) for r in results)
    
    print(f"Progress: {current}/{total} ({pct:.1f}%)")
    print(f"Last: Q{q_id} - success={success}, tokens={tokens}")
    print(f"")
    print(f"Running Statistics:")
    print(f"  Pass@1: {pass_at_1_count}/{current} ({100*pass_at_1_count/current:.1f}%)")
    print(f"  Execution Success: {exec_success_count}/{current} ({100*exec_success_count/current:.1f}%)")
    print(f"  Total Tokens: {total_tokens_so_far:,}")

print("Starting Baseline Inference (kg-axel)...")
print("=" * 60)
start_time = datetime.now()

for i, q in enumerate(questions):
    # Batch pause
    if i > 0 and i % CONFIG["batch_size"] == 0:
        print(f"\n[Batch pause: {CONFIG['batch_pause']}s]")
        time.sleep(CONFIG["batch_pause"])
    
    q_start = time.time()
    
    try:
        # Generate query using kg-axel
        generated_query, response = baseline.generate(
            question=q["question"],
            schema=schema
        )
        
        elapsed = time.time() - q_start
        
        # Execute query (if executor available)
        execution_success = False
        is_empty_result = False
        execution_result = None
        
        if executor:
            try:
                execution_result = executor.execute(generated_query)
                execution_success = True
                is_empty_result = len(execution_result) == 0
            except Exception as e:
                execution_success = False
                execution_result = str(e)
        
        # Simple pass@1 check (query text comparison)
        # Note: This is simplified - actual pass@1 needs output comparison
        pass_at_1 = False  # Will be computed in metrics evaluation notebook
        
        # Record result
        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "ground_truth": q["ground_truth"],
            "complexity": q["complexity"],
            "reasoning_level": q["reasoning_level"],
            "sublevel": q["sublevel"],
            "generated_query": generated_query,
            "execution_result": json.dumps(execution_result, ensure_ascii=False) if execution_result else "",
            "execution_success": execution_success,
            "is_empty_result": is_empty_result,
            "pass_at_1": pass_at_1,  # Will be computed later
            "total_tokens": response.usage["total_tokens"],
            "input_tokens": response.usage["prompt_tokens"],
            "output_tokens": response.usage["completion_tokens"],
            "elapsed_time": round(elapsed, 2)
        })
        
        update_display(
            i + 1, len(questions), q["id"], 
            execution_success, response.usage["total_tokens"]
        )
        
    except Exception as e:
        print(f"Error on Q{q['id']}: {e}")
        results.append({
            "question_id": q["id"],
            "question": q["question"],
            "ground_truth": q["ground_truth"],
            "complexity": q["complexity"],
            "reasoning_level": q["reasoning_level"],
            "sublevel": q["sublevel"],
            "generated_query": "",
            "execution_result": str(e),
            "execution_success": False,
            "is_empty_result": False,
            "pass_at_1": False,
            "total_tokens": 0,
            "input_tokens": 0,
            "output_tokens": 0,
            "elapsed_time": 0
        })
    
    # Rate limiting
    if i < len(questions) - 1:
        time.sleep(CONFIG["rate_limit_delay"])

end_time = datetime.now()
duration = str(end_time - start_time)

print(f"\n\nBaseline inference completed!")
print(f"Duration: {duration}")

## 8. Save Results

In [ ]:
import pandas as pd

# Create DataFrame
df = pd.DataFrame(results)

# Save to CSV
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")

## 9. Quick Summary

In [ ]:
print("=" * 60)
print("BASELINE INFERENCE SUMMARY (kg-axel)")
print("=" * 60)

total = len(results)
exec_success = sum(1 for r in results if r["execution_success"])
empty_results = sum(1 for r in results if r["is_empty_result"])
total_tokens = sum(r["total_tokens"] for r in results)
total_time = sum(r["elapsed_time"] for r in results)

print(f"\nConfiguration:")
print(f"  Model: {CONFIG['model']}")
print(f"  Prompt: {CONFIG['prompt_type']}")
print(f"  Schema: {CONFIG['schema_type']}")

print(f"\nExecution Statistics:")
print(f"  Total questions: {total}")
print(f"  Execution success: {exec_success}/{total} ({100*exec_success/total:.1f}%)")
print(f"  Empty results: {empty_results}/{total} ({100*empty_results/total:.1f}%)")

print(f"\nCost & Performance:")
print(f"  Total tokens: {total_tokens:,}")
print(f"  Avg tokens/question: {total_tokens/total:,.0f}")
print(f"  Total time: {total_time:.1f}s")
print(f"  Avg time/question: {total_time/total:.1f}s")
print(f"  Duration: {duration}")

print(f"\nOutput: {OUTPUT_PATH}")

print(f"\n" + "=" * 60)
print("Next: Run 02_inference_multiagent.ipynb")
print("Then: Run 03_metrics_evaluation.ipynb to compute Pass@k")
print("=" * 60)

## 10. Preview Results

In [ ]:
# Preview results
display(df[["question_id", "complexity", "sublevel", 
            "execution_success", "is_empty_result"]].head(10))